02 - Data Cleaning

Purpose: Document cleaning decisions, missing value handling, timestamp conversion, and duplicate checks. This file will help us understand:


# Importing Libraries


In [1]:
import pandas as pd
from pathlib import Path

# Setup


In [3]:
import sys

sys.path.append("..")
from src.ingest import (
    get_raw_files,
    load_listening_events_sample,
    load_user_profile,
    get_file_size_mb,
    profile_listening_events_in_chunks,
)
from src.clean import clean_listening_events, clean_user_profile
from src.config import PROCESSED_DATA_DIR, CLEANED_CHUNKS_DIR

# Load Raw Data


In [3]:
profile_df = load_user_profile()
profile_df.head()

,user_id,gender,age,country,signup_date
0,user_000001,m,NaN,Japan,"Aug 13, 2006"
1,user_000002,f,NaN,Peru,"Feb 24, 2006"
2,user_000003,m,22.0,United States,"Oct 30, 2005"
3,user_000004,f,NaN,NaN,"Apr 26, 2006"
4,user_000005,m,NaN,Bulgaria,"Jun 29, 2006"


In [4]:
events_df = load_listening_events_sample(100_000)
events_df.head()

,user_id,timestamp,artist_id,artist_name,track_id,track_name
0,user_000001,2009-05-04T23:08:57Z,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,NaN,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007
1,user_000001,2009-05-04T13:54:10Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Composition 0919 (Live_2009_4_15)
2,user_000001,2009-05-04T13:52:04Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc2 (Live_2009_4_15)
3,user_000001,2009-05-04T13:42:52Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Hibari (Live_2009_4_15)
4,user_000001,2009-05-04T13:42:11Z,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,NaN,Mc1 (Live_2009_4_15)


## shape


In [5]:
events_df.shape

(100000, 6)

In [6]:
profile_df.shape

(992, 5)

# Clean User Profile Data


## Clean


In [7]:
profile_clean = clean_user_profile(profile_df=profile_df)
profile_clean.head()

,user_id,gender,age,country,signup_date
0,user_000001,m,NaN,Japan,2006-08-13
1,user_000002,f,NaN,Peru,2006-02-24
2,user_000003,m,22.0,United States,2005-10-30
3,user_000004,f,NaN,<NA>,2006-04-26
4,user_000005,m,NaN,Bulgaria,2006-06-29


## shape


In [8]:
profile_clean.shape

(992, 5)

## info


In [9]:
profile_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 992 entries, 0 to 991
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   user_id      992 non-null    string        
 1   gender       884 non-null    string        
 2   age          286 non-null    float64       
 3   country      907 non-null    string        
 4   signup_date  984 non-null    datetime64[us]
dtypes: datetime64[us](1), float64(1), string(3)
memory usage: 59.5 KB


## Missing values


In [10]:
profile_clean.isna().sum()

user_id          0
gender         108
age            706
country         85
signup_date      8
dtype: int64

# Clean Listening Events Sample


## Clean


In [11]:
events_clean = clean_listening_events(events_df)
events_clean.head()

,user_id,timestamp,artist_id,artist_name,track_id,track_name,artist_key,track_key
0,user_000001,2009-05-04 23:08:57+00:00,f1b1cf71-bd35-4e99-8624-24a6e15f133a,Deep Dish,<NA>,Fuck Me Im Famous (Pacha Ibiza)-09-28-2007,f1b1cf71-bd35-4e99-8624-24a6e15f133a,name:deep_dish::fuck_me_im_famous_(pacha_ibiza...
1,user_000001,2009-05-04 13:54:10+00:00,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,<NA>,Composition 0919 (Live_2009_4_15),a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,name:坂本龍一::composition_0919_(live_2009_4_15)
2,user_000001,2009-05-04 13:52:04+00:00,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,<NA>,Mc2 (Live_2009_4_15),a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,name:坂本龍一::mc2_(live_2009_4_15)
3,user_000001,2009-05-04 13:42:52+00:00,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,<NA>,Hibari (Live_2009_4_15),a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,name:坂本龍一::hibari_(live_2009_4_15)
4,user_000001,2009-05-04 13:42:11+00:00,a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,坂本龍一,<NA>,Mc1 (Live_2009_4_15),a7f7df4a-77d8-4f12-8acd-5c60c93f4de8,name:坂本龍一::mc1_(live_2009_4_15)


## shape


In [12]:
events_clean.shape

(100000, 8)

## info


In [14]:
events_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype              
---  ------       --------------   -----              
 0   user_id      100000 non-null  string             
 1   timestamp    100000 non-null  datetime64[us, UTC]
 2   artist_id    97229 non-null   string             
 3   artist_name  100000 non-null  string             
 4   track_id     88189 non-null   string             
 5   track_name   100000 non-null  string             
 6   artist_key   100000 non-null  string             
 7   track_key    100000 non-null  string             
dtypes: datetime64[us, UTC](1), string(7)
memory usage: 23.2 MB


## missing values


In [16]:
events_clean.isna().sum()

user_id            0
timestamp          0
artist_id       2771
artist_name        0
track_id       11811
track_name         0
artist_key         0
track_key          0
dtype: int64

# Compare Raw vs Cleaned Data


## Events data


### raw row count vs cleaned row count


In [51]:
clean_events_counts = events_clean.count()
clean_events_counts

user_id        100000
timestamp      100000
artist_id       97229
artist_name    100000
track_id        88189
track_name     100000
artist_key     100000
track_key      100000
dtype: int64

In [52]:
raw_events_counts = events_df.count()
raw_events_counts

user_id        100000
timestamp      100000
artist_id       97229
artist_name    100000
track_id        88189
track_name     100000
dtype: int64

In [ ]:
print(
    f"The row count for events data in the sample \nbefore cleaning: {len(events_df)} \nAfter cleaning: {len(events_clean)}"
)

The row count for events data 
before cleaning: 100000 
After cleaning: 100000


### rows removed


In [ ]:
print(
    f"The rows removed after cleaning in the sample is: {len(events_df) - len(events_clean)}"
)

The rows removed after cleaning in the sample is: 0


### cleaned duplicate count


In [25]:
events_clean.duplicated().sum()

np.int64(0)

### raw missing track_name vs cleaned missing track_name


In [ ]:
raw_m = events_df["track_name"].isna().sum()
clean_m = events_clean["track_name"].isna().sum()
print(
    f"In the sample: \nThe raw missing track_name count: {raw_m} \nThe Cleaned Missing track_name: {clean_m}"
)

In the sample: 
The raw missing track_name count: 0 
The Cleaned Missing track_name: 0


### raw missing artist_name vs cleaned missing artist_name


In [ ]:
raw_m = events_df["artist_name"].isna().sum()
clean_m = events_clean["artist_name"].isna().sum()
print(
    f"In the sample: \nThe raw missing artist_name count: {raw_m} \nThe Cleaned Missing artist_name: {clean_m}"
)

In the sample: 
The raw missing artist_name count: 0 
The Cleaned Missing artist_name: 0


## Profile data


### raw row count vs cleaned row count


In [29]:
profile_df.count()

user_id        992
gender         884
age            286
country        907
signup_date    984
dtype: int64

In [30]:
profile_clean.count()

user_id        992
gender         884
age            286
country        907
signup_date    984
dtype: int64

In [ ]:
print(
    f"The row count for profile data in the sample \nbefore cleaning: {len(profile_df)} \nAfter cleaning: {len(profile_clean)}"
)

The row count for profile data in the sample 
before cleaning: 992 
After cleaning: 992


### rows removed


In [32]:
profile_clean.duplicated().sum()

np.int64(0)

### cleaned duplicate count


# Validate Cleaning Rules


## Are any required fields missing after cleaning?


In [49]:
REQUIRED_EVENTS_COLUMNS = [
    "user_id",
    "timestamp",
    "artist_key",
    "track_key",
    "track_name",
    "artist_name",
]
REQUIRED_PROFILE_COLUMNS = ["user_id", "age", "country", "signup_date", "gender"]

In [34]:
for col in REQUIRED_EVENTS_COLUMNS:
    if col not in events_clean:
        raise ValueError(f"Missing column: {col}")

In [35]:
for col in REQUIRED_PROFILE_COLUMNS:
    if col not in profile_clean:
        raise ValueError(f"Missing column: {col}")

## Summary


In [55]:
summary_events = pd.DataFrame(
    {
        "raw_rows_count": [len(events_df)],
        "cleaned_rows_count": [len(events_clean)],
        "rows_removed": [len(events_df) - len(events_clean)],
        "raw_duplicates": [events_df.duplicated().sum()],
        "cleaned_duplicates": [events_clean.duplicated().sum()],
        "raw_missing_artist_name": [events_df["artist_name"].isna().sum()],
        "cleaned_missing_artist_name": [events_clean["artist_name"].isna().sum()],
        "raw_missing_track_name": [events_df["track_name"].isna().sum()],
        "cleaned_missing_track_name": [events_clean["track_name"].isna().sum()],
        "raw_missing_artist_id": [events_df["artist_id"].isna().sum()],
        "cleaned_missing_artist_id": [events_clean["artist_id"].isna().sum()],
        "raw_missing_track_id": [events_df["track_id"].isna().sum()],
        "cleaned_missing_track_id": [events_clean["track_id"].isna().sum()],
    }
)
summary_events = summary_events.transpose()
summary_events

,0
raw_rows_count,100000
cleaned_rows_count,100000
rows_removed,0
raw_duplicates,0
cleaned_duplicates,0
raw_missing_artist_name,0
cleaned_missing_artist_name,0
raw_missing_track_name,0
cleaned_missing_track_name,0
raw_missing_artist_id,2771


In [56]:
summary_profile = pd.DataFrame(
    {
        "raw_rows_count": [len(profile_df)],
        "cleaned_rows_count": [len(profile_clean)],
        "rows_removed": [len(profile_df) - len(profile_clean)],
        "raw_duplicates": [profile_df.duplicated().sum()],
        "cleaned_duplicates": [profile_clean.duplicated().sum()],
    }
)
summary_profile = summary_profile.transpose()
summary_profile

,0
raw_rows_count,992
cleaned_rows_count,992
rows_removed,0
raw_duplicates,0
cleaned_duplicates,0


## Are artist_key and track_key populated?


In [36]:
events_clean["artist_key"].head()

0    f1b1cf71-bd35-4e99-8624-24a6e15f133a
1    a7f7df4a-77d8-4f12-8acd-5c60c93f4de8
2    a7f7df4a-77d8-4f12-8acd-5c60c93f4de8
3    a7f7df4a-77d8-4f12-8acd-5c60c93f4de8
4    a7f7df4a-77d8-4f12-8acd-5c60c93f4de8
Name: artist_key, dtype: string

In [39]:
events_clean["track_key"].head()

0    name:deep_dish::fuck_me_im_famous_(pacha_ibiza...
1         name:坂本龍一::composition_0919_(live_2009_4_15)
2                      name:坂本龍一::mc2_(live_2009_4_15)
3                   name:坂本龍一::hibari_(live_2009_4_15)
4                      name:坂本龍一::mc1_(live_2009_4_15)
Name: track_key, dtype: string

In [38]:
events_clean["artist_key"].isna().sum()

np.int64(0)

In [40]:
events_clean["track_key"].isna().sum()

np.int64(0)

## Did we keep rows with missing artist_id?


In [41]:
events_clean["artist_id"].isna().sum()

np.int64(2771)

## Did we keep rows with missing track_id?


In [42]:
events_clean["track_id"].isna().sum()

np.int64(11811)

## Did duplicate rows get removed?


In [43]:
dup = len(events_df) - len(events_clean)
if dup > 0:
    print(f"Duplicated row count: {dup}")
else:
    print("There are no duplicate records or none were removed")

There are no duplicate records or none were removed


# Save Cleaned Outputs - Sample


In [ ]:
profile_data_file_name = PROCESSED_DATA_DIR / "users_clean.csv"
events_data_file_name = PROCESSED_DATA_DIR / "listening_events_clean_sample.csv"

In [47]:
profile_clean.to_csv(profile_data_file_name, index=False)
print("Profile Clean data saved successfully")

Profile Clean data saved successfully


In [48]:
events_clean.to_csv(events_data_file_name, index=False)
print("Listening events Clean data saved successfully")

Listening events Clean data saved successfully


# Validate Full Cleaned Chunk Outputs


In [ ]:
number_of_chunk_files = 0
total_cleaned_rows_chunks = 0
missing_user_id = 0
missing_artist_name = 0
missing_track_name = 0
missing_timestamp = 0
missing_artist_key = 0
missing_track_key = 0

39

In [ ]:
for f in CLEANED_CHUNKS_DIR.iterdir():
    chunk = pd.read_csv(f)
    if number_of_chunk_files == 0:
        print(
            f"First chunk shape:{chunk.shape()} \nFirst chunk columns: {chunk.columns} \nFirst chunk head: {chunk.head()}"
        )
    number_of_chunk_files += 1
    total_cleaned_rows_chunks += len(chunk)

# Cleaning Decisions and Notes

- User profile dates were converted to datetime.
- User ages were converted to numeric values.
- Listening timestamps were converted to datetime.
- Rows missing `user_id`, `timestamp`, `artist_name`, or `track_name` are removed by the cleaning function. In the 100K-row sample, no rows were removed by this rule.
- Rows with missing `artist_id` or `track_id` were retained because names can be used as fallback identifiers.
- `artist_key` and `track_key` were created to support reliable grouping in later phases.
- There is a function to remove duplicate rows as a good data engineering practice. You can't see any change here since our data do not have duplicated rows
- Only a cleaned listening sample was saved in this step. Full cleaned output will be created with chunked processing later.
